# Intra-Patient vs Inter-Patient Evaluation

This notebook evaluates all trained models on both **inter-patient** and **intra-patient** test splits,
then produces side-by-side bar plots to compare generalisation performance.

| Split | Description |
|-------|-------------|
| **Inter-patient** | Test subjects that were *never seen* during training (different patients). Measures how well the model generalises to entirely new individuals. |
| **Intra-patient** | Second temporal half of the *training* subjects. Measures how well the model captures temporal dynamics of known individuals. |

A model that scores well on both splits demonstrates strong generalisation across patients *and* time.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is on path
notebook_dir = Path().absolute()
if notebook_dir.name == "examples":
    os.chdir(notebook_dir.parent)

import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "no-latex"])
%matplotlib inline

## 1. Configuration

Choose the dataset and checkpoint directory. The notebook will automatically discover all `.pt` checkpoints matching the dataset tag.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
DATASET_TYPE   = "lsd"               # "ts_young" or "lsd"
DATA_PATH      = "data/ts_young/ts_young_TR0.72.mat"
LSD_DATA_DIR   = "data/lsd"
CHECKPOINT_DIR = "checkpoints"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Metrics to display in bar plots (subset of EVAL_METRIC_KEYS)
DISPLAY_METRICS = [
    "fc_correlation",
    "fc_mse",
    "fcd_ks",
    "phfcd_ks",
    "metastability_diff",
    "power_spectrum_distance",
    "temporal_correlation",
    "autocorr_distance",
    "phase_fc_correlation",
]

# Pretty labels for the plots
METRIC_LABELS = {
    "fc_correlation":         "FC Correlation",
    "fc_mse":                 "FC MSE",
    "fcd_ks":                 "FCD (KS)",
    "phfcd_ks":               "Phase-FCD (KS)",
    "metastability_diff":     "Metastability Diff",
    "power_spectrum_distance":"Power Spectrum Dist",
    "temporal_correlation":   "Temporal Correlation",
    "autocorr_distance":      "Autocorr Distance",
    "phase_fc_correlation":   "Phase FC Correlation",
}

# Which metrics are "higher is better" (the rest are "lower is better")
HIGHER_IS_BETTER = {"fc_correlation", "temporal_correlation", "phase_fc_correlation"}

print(f"Device: {DEVICE}")

## 2. Load Dataset & Create Data Loaders

We load the full dataset and create the same train/val/test splits used during training, giving us access to the **inter-patient** and **intra-patient** test loaders.

In [ ]:
from src.dataset import load_dataset, create_data_loaders
from src.training import HopfConfig

cfg = HopfConfig()
cfg.dataset_type = DATASET_TYPE
cfg.data_path    = DATA_PATH
cfg.lsd_data_dir = LSD_DATA_DIR
cfg.use_wandb    = False
cfg.dt_min       = 0.05

dataset = load_dataset(cfg, DEVICE)
print(f"n_rois       : {dataset.n_rois}")
print(f"n_subjects   : {dataset.timeseries.shape[0]}")
print(f"n_timepoints : {dataset.n_timepoints}")

window_size = min(100, dataset.n_timepoints // 2)
_, _, test_inter_loader, test_intra_loader = create_data_loaders(
    dataset=dataset,
    window_size=window_size,
    batch_size=cfg.batch_size,
    n_windows_per_epoch=cfg.n_windows_per_epoch,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    seed=cfg.seed,
    device=DEVICE,
)
print(f"\nwindow_size  : {window_size}")
print(f"test_inter batches : {len(test_inter_loader)}")
print(f"test_intra batches : {len(test_intra_loader)}")

## 3. Discover & Load Checkpoints

Auto-discover all checkpoints for the selected dataset and assign human-readable labels.

In [ ]:
from src.models import load_model_from_checkpoint

_MODEL_TAGS = [
    ("gnn_hopf",      "GNN Hopf"),
    ("hybrid_neural", "Hybrid+Neural"),
    ("hybrid_hopf",   "Hybrid Hopf"),
    ("nsde",          "Neural SDE"),
    ("hopf",          "Hopf"),
]

def _short_label(stem: str) -> str:
    lower = stem.lower()
    for tag, label in _MODEL_TAGS:
        if tag in lower:
            return label + (" (Grid)" if "grid" in lower else "")
    return stem

ckpt_dir = Path(CHECKPOINT_DIR)
checkpoint_paths = sorted(ckpt_dir.glob(f"*{DATASET_TYPE}*.pt"))
print(f"Found {len(checkpoint_paths)} checkpoints:")
for p in checkpoint_paths:
    print(f"  {p}")

models = {}
for ckpt_path in checkpoint_paths:
    try:
        model, mtype, _ = load_model_from_checkpoint(str(ckpt_path), device=DEVICE)
        if model.n_rois != dataset.n_rois:
            print(f"  Skipping {ckpt_path.name}: ROI mismatch ({model.n_rois} != {dataset.n_rois})")
            continue
        label = _short_label(ckpt_path.stem)
        models[label] = model
        print(f"  Loaded '{label}' ({mtype})")
    except Exception as exc:
        print(f"  Failed to load {ckpt_path.name}: {exc}")

print(f"\nLoaded {len(models)} model(s): {list(models.keys())}")

## 4. Evaluate Models on Both Test Splits

For each model we run `evaluate_model_loader_metrics` on both the inter-patient and intra-patient loaders, collecting the mean and standard deviation of every metric across batches.

In [ ]:
from src.utils.evaluation import evaluate_model_loader_metrics, format_metrics_mean_std

inter_results = {}  # model_name -> {metric: value, metric_std: value}
intra_results = {}

for name, model in models.items():
    print(f"Evaluating {name} ...")
    inter_results[name] = evaluate_model_loader_metrics(
        model, test_inter_loader, cfg, n_steps=window_size, return_std=True,
    )
    intra_results[name] = evaluate_model_loader_metrics(
        model, test_intra_loader, cfg, n_steps=window_size, return_std=True,
    )
    print(f"  Inter: {format_metrics_mean_std(inter_results[name])}")
    print(f"  Intra: {format_metrics_mean_std(intra_results[name])}")
    print()

print("Evaluation complete.")

## 5. Summary Table

Quick numerical overview before plotting.

In [ ]:
import pandas as pd

rows = []
for name in models:
    for split_name, results in [("Inter", inter_results), ("Intra", intra_results)]:
        row = {"Model": name, "Split": split_name}
        for m in DISPLAY_METRICS:
            val = results[name].get(m, float("nan"))
            std = results[name].get(f"{m}_std", float("nan"))
            row[METRIC_LABELS.get(m, m)] = f"{val:.4f} +/- {std:.4f}"
        rows.append(row)

df_summary = pd.DataFrame(rows)
df_summary.set_index(["Model", "Split"], inplace=True)
df_summary

## 6. Grouped Bar Plots — Inter vs Intra per Metric

Each subplot shows one metric. For every model, two bars are drawn side-by-side:
the **blue** bar is the inter-patient score and the **orange** bar is the intra-patient score.
Error bars show the standard deviation across batches.

Metrics where **higher is better** (FC Correlation, Temporal Correlation, Phase FC Correlation)
are annotated with an upward arrow; all others are **lower is better**.

In [ ]:
from matplotlib.gridspec import GridSpec

model_names = list(models.keys())
n_models = len(model_names)
n_metrics = len(DISPLAY_METRICS)

# ── Colour palette ───────────────────────────────────────────────────────────
INTER_COLOR = "#4C72B0"   # steel blue
INTRA_COLOR = "#DD8452"   # warm orange

# ── Layout: 3 columns, as many rows as needed ───────────────────────────────
n_cols = 3
n_rows = int(np.ceil(n_metrics / n_cols))

fig = plt.figure(figsize=(4.5 * n_cols, 3.2 * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, hspace=0.55, wspace=0.35)

bar_width = 0.32
x = np.arange(n_models)

for idx, metric_key in enumerate(DISPLAY_METRICS):
    row, col = divmod(idx, n_cols)
    ax = fig.add_subplot(gs[row, col])

    inter_means = [inter_results[m].get(metric_key, 0) for m in model_names]
    intra_means = [intra_results[m].get(metric_key, 0) for m in model_names]
    inter_stds  = [inter_results[m].get(f"{metric_key}_std", 0) for m in model_names]
    intra_stds  = [intra_results[m].get(f"{metric_key}_std", 0) for m in model_names]

    bars_inter = ax.bar(
        x - bar_width / 2, inter_means, bar_width,
        yerr=inter_stds, label="Inter-patient",
        color=INTER_COLOR, edgecolor="white", linewidth=0.6,
        capsize=3, error_kw=dict(lw=0.8, capthick=0.8),
        zorder=3, alpha=0.9,
    )
    bars_intra = ax.bar(
        x + bar_width / 2, intra_means, bar_width,
        yerr=intra_stds, label="Intra-patient",
        color=INTRA_COLOR, edgecolor="white", linewidth=0.6,
        capsize=3, error_kw=dict(lw=0.8, capthick=0.8),
        zorder=3, alpha=0.9,
    )

    # Title with direction arrow
    direction = "higher is better" if metric_key in HIGHER_IS_BETTER else "lower is better"
    pretty_name = METRIC_LABELS.get(metric_key, metric_key)
    ax.set_title(f"{pretty_name}\n({direction})", fontsize=9, fontweight="bold", pad=6)

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_ylabel(pretty_name, fontsize=7)

    # Light grid behind bars
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
    ax.xaxis.grid(False)

    # Add value labels on top of bars
    for bars in [bars_inter, bars_intra]:
        for bar in bars:
            h = bar.get_height()
            if not np.isnan(h) and h != 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2, h,
                    f"{h:.3f}", ha="center", va="bottom",
                    fontsize=5.5, fontweight="medium",
                )

# Remove empty subplots
for idx in range(n_metrics, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    fig.add_subplot(gs[row, col]).set_visible(False)

# Single shared legend at the bottom
handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=INTER_COLOR, edgecolor="white", alpha=0.9),
    plt.Rectangle((0, 0), 1, 1, facecolor=INTRA_COLOR, edgecolor="white", alpha=0.9),
]
fig.legend(
    handles, ["Inter-patient", "Intra-patient"],
    loc="lower center", ncol=2, fontsize=9,
    frameon=True, fancybox=True, shadow=False,
    bbox_to_anchor=(0.5, -0.02),
)

fig.suptitle(
    f"Inter-Patient vs Intra-Patient Evaluation  ({DATASET_TYPE.upper()} dataset)",
    fontsize=13, fontweight="bold", y=1.01,
)

plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_barplots.pdf", bbox_inches="tight", dpi=200)
plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_barplots.png", bbox_inches="tight", dpi=200)
plt.show()
print("Saved to paper/images/")

## 7. Radar / Spider Chart — Normalised Profile

To compare model profiles at a glance, we normalise each metric to `[0, 1]` across all models
(flipping "lower is better" metrics so that *outward = better* for every axis)
and overlay them on a radar chart.

In [ ]:
from matplotlib.patches import FancyBboxPatch

def _normalise_metrics(results_dict, metric_keys, higher_is_better_set):
    """Normalise metrics to [0,1] across models; flip lower-is-better so outward = better."""
    raw = {m: {mk: results_dict[m].get(mk, 0) for mk in metric_keys} for m in results_dict}
    normed = {m: {} for m in results_dict}
    for mk in metric_keys:
        vals = [raw[m][mk] for m in results_dict]
        lo, hi = min(vals), max(vals)
        rng = hi - lo if hi != lo else 1.0
        for m in results_dict:
            v = (raw[m][mk] - lo) / rng
            normed[m][mk] = v if mk in higher_is_better_set else 1.0 - v
    return normed

# Build normalised scores for both splits
normed_inter = _normalise_metrics(inter_results, DISPLAY_METRICS, HIGHER_IS_BETTER)
normed_intra = _normalise_metrics(intra_results, DISPLAY_METRICS, HIGHER_IS_BETTER)

# Radar chart
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon
labels = [METRIC_LABELS.get(m, m) for m in DISPLAY_METRICS]

# Distinct colour per model
cmap = plt.cm.get_cmap("tab10", n_models)
model_colors = {name: cmap(i) for i, name in enumerate(model_names)}

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), subplot_kw=dict(polar=True))

for ax, (split_label, normed) in zip(axes, [("Inter-Patient", normed_inter), ("Intra-Patient", normed_intra)]):
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=7)

    for name in model_names:
        vals = [normed[name][mk] for mk in DISPLAY_METRICS] + [normed[name][DISPLAY_METRICS[0]]]
        ax.plot(angles, vals, linewidth=1.5, label=name, color=model_colors[name])
        ax.fill(angles, vals, alpha=0.08, color=model_colors[name])

    ax.set_ylim(0, 1.05)
    ax.set_title(split_label, fontsize=11, fontweight="bold", pad=20)
    ax.tick_params(axis="y", labelsize=6)

# Shared legend
handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(
    handles, lbls, loc="lower center", ncol=min(n_models, 4),
    fontsize=8, frameon=True, bbox_to_anchor=(0.5, -0.04),
)
fig.suptitle(
    f"Normalised Metric Profiles  ({DATASET_TYPE.upper()})\n(outward = better)",
    fontsize=12, fontweight="bold", y=1.04,
)

plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_radar.pdf", bbox_inches="tight", dpi=200)
plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_radar.png", bbox_inches="tight", dpi=200)
plt.show()

## 8. Delta Plot — Intra minus Inter Gap

How much does each model's performance change between the two splits?
Positive delta means the model is *better* on the intra-patient split; negative means it
generalises better to unseen patients. Bars are coloured by sign.

In [ ]:
fig = plt.figure(figsize=(4.5 * n_cols, 3.2 * n_rows))
gs = GridSpec(n_rows, n_cols, figure=fig, hspace=0.55, wspace=0.35)

BETTER_COLOR = "#2ca02c"  # green  = intra is better
WORSE_COLOR  = "#d62728"  # red    = inter is better

for idx, metric_key in enumerate(DISPLAY_METRICS):
    row, col = divmod(idx, n_cols)
    ax = fig.add_subplot(gs[row, col])

    inter_vals = np.array([inter_results[m].get(metric_key, 0) for m in model_names])
    intra_vals = np.array([intra_results[m].get(metric_key, 0) for m in model_names])

    # For "higher is better" metrics, positive delta (intra > inter) is good
    # For "lower is better" metrics, negative delta (intra < inter) is good
    delta = intra_vals - inter_vals

    # Colour: green when intra is "better", red otherwise
    if metric_key in HIGHER_IS_BETTER:
        colors = [BETTER_COLOR if d > 0 else WORSE_COLOR for d in delta]
    else:
        colors = [BETTER_COLOR if d < 0 else WORSE_COLOR for d in delta]

    ax.bar(x, delta, width=0.55, color=colors, edgecolor="white", linewidth=0.6, zorder=3, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.6, zorder=2)

    pretty_name = METRIC_LABELS.get(metric_key, metric_key)
    direction = "higher is better" if metric_key in HIGHER_IS_BETTER else "lower is better"
    ax.set_title(f"{pretty_name}\n({direction})", fontsize=9, fontweight="bold", pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=35, ha="right", fontsize=7)
    ax.tick_params(axis="y", labelsize=7)
    ax.set_ylabel("Intra - Inter", fontsize=7)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
    ax.xaxis.grid(False)

    # Value annotations
    for i, d in enumerate(delta):
        va = "bottom" if d >= 0 else "top"
        ax.text(i, d, f"{d:+.3f}", ha="center", va=va, fontsize=5.5, fontweight="medium")

# Remove empty subplots
for idx in range(n_metrics, n_rows * n_cols):
    row, col = divmod(idx, n_cols)
    fig.add_subplot(gs[row, col]).set_visible(False)

# Legend
handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor=BETTER_COLOR, alpha=0.85),
    plt.Rectangle((0, 0), 1, 1, facecolor=WORSE_COLOR, alpha=0.85),
]
fig.legend(
    handles, ["Intra better", "Inter better"],
    loc="lower center", ncol=2, fontsize=9,
    frameon=True, fancybox=True, bbox_to_anchor=(0.5, -0.02),
)
fig.suptitle(
    f"Generalisation Gap: Intra - Inter  ({DATASET_TYPE.upper()} dataset)",
    fontsize=13, fontweight="bold", y=1.01,
)

plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_delta.pdf", bbox_inches="tight", dpi=200)
plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_delta.png", bbox_inches="tight", dpi=200)
plt.show()

## 9. Horizontal Grouped Bar — Compact Single-Figure Summary

A condensed horizontal bar chart with all models and metrics in a single figure, ideal for paper appendix or quick comparison.

In [ ]:
# ── Compact horizontal grouped bar chart ─────────────────────────────────────
fig, axes = plt.subplots(
    1, n_models, figsize=(4 * n_models, 0.45 * n_metrics + 1.2),
    sharey=True,
)
if n_models == 1:
    axes = [axes]

y = np.arange(n_metrics)
bar_h = 0.35

for ax, name in zip(axes, model_names):
    inter_vals = [inter_results[name].get(mk, 0) for mk in DISPLAY_METRICS]
    intra_vals = [intra_results[name].get(mk, 0) for mk in DISPLAY_METRICS]
    inter_stds = [inter_results[name].get(f"{mk}_std", 0) for mk in DISPLAY_METRICS]
    intra_stds = [intra_results[name].get(f"{mk}_std", 0) for mk in DISPLAY_METRICS]

    ax.barh(y - bar_h / 2, inter_vals, bar_h, xerr=inter_stds,
            color=INTER_COLOR, alpha=0.9, edgecolor="white", linewidth=0.5,
            capsize=2, error_kw=dict(lw=0.6), label="Inter", zorder=3)
    ax.barh(y + bar_h / 2, intra_vals, bar_h, xerr=intra_stds,
            color=INTRA_COLOR, alpha=0.9, edgecolor="white", linewidth=0.5,
            capsize=2, error_kw=dict(lw=0.6), label="Intra", zorder=3)

    ax.set_yticks(y)
    ax.set_yticklabels([METRIC_LABELS.get(m, m) for m in DISPLAY_METRICS], fontsize=7)
    ax.set_title(name, fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelsize=6)
    ax.set_axisbelow(True)
    ax.xaxis.grid(True, linestyle="--", alpha=0.4, linewidth=0.5)
    ax.yaxis.grid(False)
    ax.invert_yaxis()

axes[-1].legend(fontsize=7, loc="lower right")
fig.suptitle(
    f"Per-Model Metric Comparison  ({DATASET_TYPE.upper()})",
    fontsize=12, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_horizontal.pdf", bbox_inches="tight", dpi=200)
plt.savefig(f"paper/images/{DATASET_TYPE}/intra_vs_inter_horizontal.png", bbox_inches="tight", dpi=200)
plt.show()

## 10. Save Results to JSON

Persist the raw metrics so they can be reloaded without re-running evaluation.

In [ ]:
import json

output = {
    "test_inter": {
        name: {k: float(v) for k, v in metrics.items()}
        for name, metrics in inter_results.items()
    },
    "test_intra": {
        name: {k: float(v) for k, v in metrics.items()}
        for name, metrics in intra_results.items()
    },
}

output_path = Path(f"results/{DATASET_TYPE}_intra_vs_inter_results.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w") as f:
    json.dump(output, f, indent=2, sort_keys=True)

print(f"Results saved to {output_path}")